# Kisara — Gambar gratis di Colab (FLUX.1 schnell / SDXL)

Notebook ini menggambar **semua gambar klip** Kisara memakai GPU gratis Colab (T4, 16 GB).
Modelnya open-source dan berjalan penuh — jadi kualitasnya di atas layanan gratis yang dibatasi.

**Cara pakai**
1. Di Kisara: buka proyek → langkah **Storyboard** → klik **Ekspor prompt**.
   Simpan file `...-prompts.json`.
2. Di Colab: **Runtime → Change runtime type → T4 GPU**, lalu **Save**.
3. Jalankan sel dari atas ke bawah. Saat diminta, unggah file JSON tadi.
4. Unduh ZIP hasilnya, lalu di Kisara pakai tombol **Impor gambar** per klip.

> **Nama file = urutan klip.** `001.png` untuk klip 1, `002.png` untuk klip 2, dan seterusnya,
> jadi mudah dipasangkan saat mengimpor.
>
> Notebook ini dipakai sebagai notebook interaktif — itu memang cara pakai Colab yang diperbolehkan.
> Menjadikannya server/backend untuk aplikasi lain tidak diperbolehkan oleh aturan Colab.

## 1. Pastikan dapat GPU
Kalau tidak muncul `Tesla T4`, ubah runtime ke GPU lewat menu Runtime.

In [ ]:
!nvidia-smi

## 2. Pasang dependensi

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece protobuf

## 3. Unggah file prompt dari Kisara
Pilih file `...-prompts.json` yang dibuat tombol **Ekspor prompt**.

In [ ]:
import json
from pathlib import Path
from google.colab import files

uploaded = files.upload()
name = next(iter(uploaded))
data = json.loads(Path(name).read_text(encoding='utf-8'))

ASPECT = data.get('aspect', '16:9')
JOBS = data['images']

print(f"Proyek: {data.get('project', '(tanpa judul)')} · rasio {ASPECT} · {len(JOBS)} gambar")
for job in JOBS[:3]:
    print(f"  [{job['index']}] {job['prompt'][:100]}...")
if len(JOBS) > 3:
    print(f'  ... dan {len(JOBS) - 3} lagi')

## 4. Pilih model dan ukuran

- **`flux`** — FLUX.1 schnell. Kualitas paling bagus, 4 langkah. Di T4 agak lambat (~1–3 menit/gambar)
  karena memori di-offload ke RAM.
- **`sdxl`** — Stable Diffusion XL. Jauh lebih cepat (~20–40 detik/gambar), kualitas sedang.

Ukuran mengikuti rasio video dan harus kelipatan 16.

In [ ]:
MODEL = 'flux'          # 'flux' atau 'sdxl'
STEPS = 4 if MODEL == 'flux' else 30

OUT_DIR = Path('/content/out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if ASPECT == '9:16':
    WIDTH, HEIGHT = 576, 1024
else:
    WIDTH, HEIGHT = 1024, 576

print(f'{MODEL} · {WIDTH}x{HEIGHT} · {STEPS} langkah')

## 5. Muat model
Unduhan pertama beberapa GB. `enable_model_cpu_offload()` yang membuat model besar tetap muat di T4 16 GB.

In [ ]:
import torch

if MODEL == 'flux':
    from diffusers import FluxPipeline

    pipe = FluxPipeline.from_pretrained('black-forest-labs/FLUX.1-schnell', torch_dtype=torch.bfloat16)
    GUIDANCE = 0.0          # schnell tidak memakai classifier-free guidance
else:
    from diffusers import StableDiffusionXLPipeline

    pipe = StableDiffusionXLPipeline.from_pretrained(
        'stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16, variant='fp16'
    )
    GUIDANCE = 7.0

pipe.enable_model_cpu_offload()
print('model siap')

## 6. Gambar semuanya
Gambar yang sudah ada akan dilewati, jadi aman dijalankan ulang kalau sesi Colab terputus.

In [ ]:
import gc
import time

done = []
for job in JOBS:
    out = OUT_DIR / f"{job['index']:03d}.png"
    if out.exists():
        print(f"[{job['index']}/{len(JOBS)}] lewati (sudah ada)")
        done.append(out)
        continue

    print(f"[{job['index']}/{len(JOBS)}] menggambar...")
    started = time.time()
    image = pipe(
        prompt=job['prompt'],
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=WIDTH,
        height=HEIGHT,
    ).images[0]
    image.save(out)
    done.append(out)
    print(f"    selesai dalam {time.time() - started:.0f} detik")
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nSelesai: {len(done)} gambar di {OUT_DIR}')

## 7. Unduh hasilnya
Ekstrak ZIP-nya di komputer, lalu di Kisara buka tiap klip → **Impor gambar** dan pilih file
sesuai nomornya.

In [ ]:
import os
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/kisara-images', 'zip', OUT_DIR)
print('Ukuran:', round(os.path.getsize(zip_path) / 1e6, 1), 'MB')
files.download(zip_path)

## Kalau ada masalah

**CUDA out of memory** — ganti `MODEL = 'sdxl'` di sel 4, jalankan ulang sel 4–6.
SDXL jauh lebih ringan daripada FLUX.

**Sesi Colab mati di tengah jalan** — jalankan ulang sel 4–6. Gambar yang sudah jadi dilewati,
jadi tinggal lanjut dari yang belum.

**Tidak dapat GPU** — Colab gratis membatasi pemakaian GPU secara dinamis. Tunggu beberapa jam,
atau jalankan di waktu yang lebih sepi.

**Karakter terlihat berbeda antar klip** — notebook ini menggambar dari teks saja, tanpa gambar
referensi. Supaya tokohnya konsisten, pastikan deskripsi tokoh di Kisara cukup rinci
(langkah Ide cerita, bagian pemeran).